In [1]:
import polars as pl

df = pl.read_parquet(
    "datasets/dataset_merged.parquet",
)


DATASET_NAME_PERFIX = "datasets/dataset_split_"

df


original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source
str,str,str,f64,bool,str
"""SAGGSEPSQADLDALLSAVRDNRLSIEQAV…","""SAGGSEPSQADLDALLLAVRDNRLSIEQAV…","""S17L""",0.005029,false,"""megascale"""
"""SAGGSQIFVKTLTGKTITLEVEPSDTIENV…","""SAGGSQIFVKTLTGKDITLEVEPSDTIENV…","""T16D""",-0.17017,false,"""megascale"""
"""SAGGSAGGSAGGWEITFERNGKRITVRTTD…","""SAGGSAGGSAGGWEITFERNGKRITVRTTD…","""L36D""",-0.078552,false,"""megascale"""
"""SAGGSAGGSAGGSTVKVRLGHLEVTLHNVS…","""SAGGSAGGSAGGSTVKVRLGHLEVTLHNVS…","""E55K""",-0.007361,false,"""megascale"""
"""SAGGSAGGSAGGSVLKALERTRQLDIPDEK…","""SAGGSAGGSAGGSVLKALERTRQLDIPDEK…","""L39G""",-0.334433,false,"""megascale"""
…,…,…,…,…,…
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""V1059C""",-0.449371,false,"""lehner"""
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""W1059C""",-0.373695,false,"""lehner"""
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""Y1059C""",-0.461689,false,"""lehner"""


In [2]:
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0
HOLDOUT_RATIO = VAL_RATIO + TEST_RATIO

# Získání celkového počtu řádků
n_rows = len(df)
print(f"Celkový počet řádků v datasetu: {n_rows}")

# 1. Spočítáme frekvenci každé unikátní sekvence
sequence_counts = df.group_by("original_seq_full").len().rename({"len": "count"})

# 2. Seřadíme sekvence od nejčastějších po nejméně časté
sequences_sorted_by_freq = sequence_counts.sort("count", descending=True)

# Získáme seřazený seznam názvů sekvencí
sorted_unique_sequences = sequences_sorted_by_freq.get_column("original_seq_full").to_list()

print("\nNejčastěji zastoupené sekvence:")
print(sequences_sorted_by_freq.head(5))


# 3. Postupný výběr sekvencí pro holdout sadu (validace + test)
holdout_sequences = []
current_holdout_size = 0
target_holdout_size = int(HOLDOUT_RATIO * n_rows)

# Iterujeme přes SEŘAZENÝ seznam (od nejčastějších)
for seq in sorted_unique_sequences:
    # Přidáme další sekvenci do seznamu pro holdout
    holdout_sequences.append(seq)

    # Zjistíme, kolik řádků v datasetu odpovídá dosud vybraným sekvencím
    size_so_far = df.filter(pl.col("original_seq_full").is_in(holdout_sequences)).height
    # Pokud jsme dosáhli nebo překročili cílovou velikost, ukončíme cyklus
    if size_so_far >= target_holdout_size:
        break

print(f"\nVybráno {len(holdout_sequences)} nejčastějších unikátních sekvencí pro validaci a testování.")

# 4. Rozdělení datasetu na trénovací a "holdout" část
train_df = df.filter(
    ~pl.col("original_seq_full").is_in(holdout_sequences)
)

holdout_pool_df = df.filter(
    pl.col("original_seq_full").is_in(holdout_sequences)
)

# 5. Rozdělení "holdout" části na validační a testovací sady
# Zamícháme data, aby bylo rozdělení náhodné
holdout_pool_df = holdout_pool_df.sample(fraction=1, shuffle=True, seed=123)

# Velikost validační sady je 10 % z *původního celkového počtu*
val_size = int(VAL_RATIO * n_rows)

val_df = holdout_pool_df.slice(0, val_size)
# Testovací sada je zbytek z holdout dat
test_df = holdout_pool_df.slice(val_size)

# (Volitelné) Vytvoření testovacího setu bez reverzních mutací
test_df_noreverse = test_df.filter(pl.col("reverse") == False)

# --- Konec úprav ---


# Uložení každého datasetu do samostatného CSV souboru
train_df.write_csv(f"{DATASET_NAME_PERFIX}train.csv")
val_df.write_csv(f"{DATASET_NAME_PERFIX}validation.csv")
test_df.write_csv(f"{DATASET_NAME_PERFIX}test.csv")
test_df_noreverse.write_csv(f"{DATASET_NAME_PERFIX}noreverse_test.csv")

# Výpis finálních statistik
print("\n--- Výsledky rozdělení ---")
print(f"Trénovací sada:   {len(train_df):>6} řádků ({len(train_df)/n_rows:>6.1%})")
print(f"Validační sada:    {len(val_df):>6} řádků ({len(val_df)/n_rows:>6.1%})")
print(f"Testovací sada:     {len(test_df):>6} řádků ({len(test_df)/n_rows:>6.1%})")
print("-------------------------------")
print(f"Celkem zpracováno: {len(train_df) + len(val_df) + len(test_df):>6} řádků")
print(f"\nSoubory byly úspěšně uloženy s prefixem '{DATASET_NAME_PERFIX}'.")

Celkový počet řádků v datasetu: 1949832

Nejčastěji zastoupené sekvence:
shape: (5, 2)
┌─────────────────────────────────┬───────┐
│ original_seq_full               ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ MLEAIDKNRALHAAERLQTKLRERGDVANE… ┆ 7077  │
│ MAERGGDGGESERFNPGELRMAQQQALRFR… ┆ 5734  │
│ MSKSLKKKSHWTSKVHESVIGRNPEGQLGF… ┆ 5505  │
│ MDCLCIVTTKKYRYQDEDTPPLEHSPAHLP… ┆ 5396  │
│ MRPGTGAERGGLMVSEMESHPPSQGPGDGE… ┆ 5083  │
└─────────────────────────────────┴───────┘

Vybráno 64 nejčastějších unikátních sekvencí pro validaci a testování.

--- Výsledky rozdělení ---
Trénovací sada:   1754532 řádků ( 90.0%)
Validační sada:    194983 řádků ( 10.0%)
Testovací sada:        317 řádků (  0.0%)
-------------------------------
Celkem zpracováno: 1949832 řádků

Soubory byly úspěšně uloženy s prefixem 'datasets/dataset_split_'.


In [3]:
train_df.filter(pl.col("reverse") == True).describe()

statistic,original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source
str,str,str,str,f64,f64,str
"""count""","""974916""","""974916""","""974916""",974916.0,974916.0,"""974916"""
"""null_count""","""0""","""0""","""0""",0.0,0.0,"""0"""
"""mean""",null,null,null,0.236417,1.0,null
"""std""",null,null,null,0.268923,null,null
"""min""","""*DPFLVLLHSVSSSLSSSELTELKFLCLGR…","""DTINITLPDGKTLTLTVTPEFTVKELAEEI…","""A10*""",-0.986593,1.0,"""lehner"""
"""25%""",null,null,null,0.028976,null,null
"""50%""",null,null,null,0.159067,null,null
"""75%""",null,null,null,0.405425,null,null
"""max""","""YVIRSIIKSSRLEEDRKRYLMTLLDDIKGA…","""SVPQRAWTVEQLRSEQLPKKDIIKFLQEHG…","""Y9W:R37Y""",0.999982,1.0,"""megascale"""


In [4]:
df.filter(pl.col("original_seq_full").is_in(holdout_sequences))

original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source
str,str,str,f64,bool,str
"""SAGGTYTWNTKEEAKQAFKELLKEKRVPSN…","""SAGGTYTWNTKEEAKQAFKELLKDKRVPSN…","""E24D:R45Q""",-0.242959,false,"""megascale"""
"""SAGGSPAHPYDRLKTTSTDPVSDIDVTRRE…","""SAGGSPAHPYDSLKTTSTYPVSDIDVTRRE…","""R12S:D19Y""",-0.039124,false,"""megascale"""
"""SSGGSSILDRAVIEHNLLSASKLYNNITFE…","""SSGGSSILDRAVIIHNLLSASKLYNNITFE…","""E14I:R55A""",-0.097891,false,"""megascale"""
"""SAGGSDAPDEFRDPLMDTLMTDPVRLPSGT…","""SAGGSDAPDEFRDPLMDTLMTDPVRLPSGT…","""E60A""",-0.077887,false,"""megascale"""
"""SAGGSAGGSPWSAKENKAFERALAVYDKDT…","""SAGGSAGGSPWSAKNNKAFERALAVYDKDT…","""E15N:T45N""",-0.453726,false,"""megascale"""
…,…,…,…,…,…
"""MVDYIVEYDYDAVHDDELTIRVGEIIRNVK…","""MVDYIVEYDYDAVHDDELTIRVGEIIRNVK…","""S109K""",-0.10794,false,"""lehner"""
"""MVDYIVEYDYDAVHDDELTIRVGEIIRNVK…","""MVDYIVEYDYDAVHDDELTIRVGEIIRNVK…","""T109K""",0.123563,false,"""lehner"""
"""MVDYIVEYDYDAVHDDELTIRVGEIIRNVK…","""MVDYIVEYDYDAVHDDELTIRVGEIIRNVK…","""V109K""",-0.158044,false,"""lehner"""
